In [1]:
!pip -q install -U tokenizers tqdm transformers


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [47]:
import os, math, time, json, random
import numpy as np
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import PreTrainedTokenizerFast

In [48]:
# load custom tokenizer
DIR_TOKENIZER = "../checkpoints_lightning/final/tokenizer"
hf_tokenizer = PreTrainedTokenizerFast.from_pretrained(DIR_TOKENIZER)

In [49]:
len(hf_tokenizer)

26000

In [ ]:
# class DiffusionTransformer(nn.Module):
#     def __init__(self, cfg):
#         super().__init__()

#         self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
#         self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
#         self.time_emb = nn.Embedding(cfg["diffusion_steps"] + 1, cfg["emb_dim"])
#         self.drop = nn.Dropout(cfg["dropout"])
#         encoder_layer = nn.TransformerEncoderLayer(
#             d_model=cfg["emb_dim"],
#             nhead=cfg["n_heads"],
#             dim_feedforward=cfg["d_ff"],
#             dropout=cfg["dropout"],
#             activation="gelu",
#             batch_first=True,
#             norm_first=False
#         )

#         self.encoder = nn.TransformerEncoder(encoder_layer , num_layers=cfg["n_layers"])
#         self.final_norm = nn.LayerNorm(cfg["emb_dim"])
#         self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

#         # Tie weights (optional; common in LMs)
#         self.out_head.weight = self.tok_emb.weight

#     def forward(self, input_ids, timesteps, attention_mask=None):
#         # input_ids of size [batch_size, seq_length]
#         # timesteps of size [batch_size] integer diffusion step in {1, 2, ...T}
#         batch_size, seq_length = input_ids.shape

#         tok_embeds = self.tok_emb(input_ids)
#         pos_embeds = self.pos_emb(torch.arange(seq_length, device=input_ids.device))
#         x = tok_embeds + pos_embeds # [batch_size, seq_length, emb_dim] emb_dim = d_model

#         # NEW
#         t_emb = self.time_emb(timesteps) # [batch_size, emb_dim]
#         t_emb = t_emb.unsqueeze(1) # [batch_size, 1, emb_dim] to allow broadcasting later
#         x = x + t_emb # [batch_size, seq_length, emb_dim]

#         x = self.drop(x)

#         if attention_mask is None:
#             src_key_padding_mask = None
#         else:
#             src_key_padding_mask = ~attention_mask  # invert: True = PAD tokens which will be ignored

#         x = self.encoder(x, is_causal=False, src_key_padding_mask=src_key_padding_mask) # this is by default False the causal attention mask
#         x = self.final_norm(x)
#         logits = self.out_head(x) # [batch_size, seq_length, vocab_size]

#         return logits

In [50]:
class DiffusionTransformerLM(nn.Module):
    def __init__(self, cfg: dict):
        super().__init__()
        self.cfg = cfg

        d_model = cfg["emb_dim"]
        seq_len = cfg["context_length"]

        self.tok_emb = nn.Embedding(cfg["vocab_size"], d_model)
        self.pos_emb = nn.Embedding(seq_len, d_model)
        self.time_emb = nn.Embedding(cfg["diffusion_steps"] + 1, d_model)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=cfg["n_heads"],
            dim_feedforward=cfg["d_ff"],
            dropout=cfg["dropout"],
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )

        self.encoder = nn.TransformerEncoder(
            enc_layer,
            num_layers=cfg["n_layers"]
        )

        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, cfg["vocab_size"], bias=False)

        # tie weights (standard LM trick)
        self.lm_head.weight = self.tok_emb.weight

        self.drop = nn.Dropout(cfg["dropout"])

    def forward(self, input_ids, timesteps, attention_mask=None):
        B, L = input_ids.shape

        if L > self.cfg["context_length"]:
            raise ValueError(
                f"Sequence length {L} > cfg['context_length'] {self.cfg['context_length']}"
            )

        # positions
        pos = torch.arange(L, device=input_ids.device).unsqueeze(0)

        x = self.tok_emb(input_ids) + self.pos_emb(pos)

        # diffusion timestep embedding
        t_emb = self.time_emb(timesteps).unsqueeze(1)
        x = x + t_emb

        x = self.drop(x)

        # padding mask handling
        if attention_mask is not None:
            src_key_padding_mask = ~attention_mask
        else:
            src_key_padding_mask = None

        x = self.encoder(x, src_key_padding_mask=src_key_padding_mask)

        x = self.ln_f(x)
        logits = self.lm_head(x)

        return logits

In [51]:
OUT_DIR = "../checkpoints_lightning/final/"
# cfg = {
#     "vocab_size": len(hf_tokenizer),
#     "context_length": 256,
#     "emb_dim": 384,
#     "n_layers": 6,
#     "n_heads": 6,
#     "d_ff": 1536, # 4*emb_dim
#     "dropout": 0.1,
#     "diffusion_steps": 64
# }

cfg = {
    "vocab_size": 26_000,
    "context_length": 256,
    "emb_dim": 512,
    "n_layers": 10,
    "n_heads": 8,
    "d_ff": 2048, # 4*emb_dim
    "dropout": 0.1,
    "diffusion_steps": 128
}
model = DiffusionTransformerLM(cfg)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
state_dict = torch.load(os.path.join(OUT_DIR, "model.pt"), map_location=device)
model.load_state_dict(state_dict)
model = model.to(device)
# compute the number of params
number_params = sum(p.numel() for p in model.parameters())
print(f"{number_params:,} params")

45,033,984 params


c:\Users\mouad\VSCode_Projects\.venv\Lib\site-packages\torch\nn\modules\transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


In [52]:
MASK_ID = hf_tokenizer.mask_token_id
# we start with a completely masked sequence
full_noise_input = torch.full((1, cfg["context_length"]), MASK_ID, device=device)

# we then append to it the user query
user_query = "Once upon a time"
prompt_text = f"<|user|>\n{user_query}.\n<|assistant|>\n<|end|>\n"
prompt_ids = hf_tokenizer.encode(prompt_text, add_special_tokens=True)
length_prompt = len(prompt_ids)
prompt_ids = torch.tensor(prompt_ids, dtype=torch.long, device=device).unsqueeze(0) # shape [1, length_prompt]

# we then replace the first tokens of the noisy input with the prompt tokens, this will be tthe input at first step s
full_noise_input[:, :length_prompt] = prompt_ids
context_ex_input = full_noise_input

# flag them as fixed tokens (won't be denoised during the inference)
fixed_tokens = torch.zeros((1, cfg["context_length"]), dtype=torch.bool, device=device)
fixed_tokens[:, :length_prompt] = True

# will give the positions of the tokens to be changed, i.e those outside of the prompt text user
update_mask = ~fixed_tokens

In [55]:
model.eval()
frames = []
x = context_ex_input.clone().to(device)

for s in range(cfg["diffusion_steps"], 0, -1): # reversed T, T-1, ...., 1, 0
    t = torch.tensor([s],dtype=torch.long, device=device)
    with torch.no_grad():
      logits = model(x, timesteps=t)

    # add a top_k logic on top of predicted logits
    topk_vals, topk_idx = torch.topk(logits, k=50, dim=-1)
    filtered = torch.full_like(logits, float("-inf"))
    filtered.scatter_(dim=-1, index=topk_idx, src=topk_vals)
    logits = filtered

    probs = torch.softmax(logits, dim=-1)   # [1, seq_length, vocab_size] as we don't look only at the last token to predict but the whole seq
    flat = probs.view(-1, probs.size(-1)) # [seq_length, vocab_size]

    sampled = torch.multinomial(flat, num_samples=1)  # (seq_length, 1)
    sampled = sampled.squeeze(-1).unsqueeze(0) # [1, seq_length]

    # get the probas associated with the tokens idx that were sampled
    confids = probs.gather(dim=-1, index=sampled.unsqueeze(-1)).squeeze(-1)

    # update only the tokens outside of the prompt user
    x = torch.where(update_mask, sampled, x)
    '''
    at each prediction, we denoise all the masked tokens from the sampled var with the where method

    but we want to iteratively denoise subset of masked tokens and keep some MASKED, until at last step s=1, the tokens
    to be keepen as MASK token are zero : i.e nbr_tokens_kept_masked = 0, so next_ratio is zero

    '''
    next_ratio = (s - 1) / cfg["diffusion_steps"]
    nbr_tokens_kept_masked = int((cfg["context_length"] - length_prompt) * next_ratio)

    # ignore prompt tokens by forcing high prob there
    conf_for_rank = confids.clone()
    conf_for_rank[:, :length_prompt] = float("inf")

    # get the lowest idx token with low probabitlity to remask again
    _, keep_idx = torch.topk(
        conf_for_rank,
        k=nbr_tokens_kept_masked,
        dim=1,
        largest=False
    )
    "important, unlike in MaskGIT, when a token is predicted, it can be masked again  in future steps, so I don't filter them out"
    # replace those token idx with the MASK ID token
    x[0, keep_idx] = MASK_ID

    decoded = hf_tokenizer.decode(x[0].tolist())
    frames.append(decoded)

generated_ids = x

In [56]:
generated_text = hf_tokenizer.decode(generated_ids[0].tolist())
print(generated_text)

[BOS]<|user|>
Once upon a time.
<|assistant|>
<|end|>
[EOS]
Write a.
<|assistant|>
There was a boy named Tim. Tim lived in the park with his mom. He liked to catch catch in the park. One day, he saw a big bird in the sky. It was a pretty bird. Tim wanted to catch with the bird, but his mom said no.

Tim was very sad. He kept catch with the bird. He felt sad. His mom said, "Tim, you can't catch with the bird. It is mine."

Tim's mom. She said, "No, Tim. You cannot play with bird. You can't have it."

Tim was sad. He wanted to catch with the bird. He He wanted the bird too. He wanted to keep the bird. He asked his mom, "Can I keep the bird?" His mom said, "No, you can keep my. You can keep. will bite you."

Tim did not listen. He did not like the bird. He wanted to hurt his mom.

Then, big bird saw the bird. The bird flew away. It flew to the tree. Tim and his mom were not happy. They played with the bird


<details>
<summary>

## Second Method insipred by MaskGIT technique for decoding

</summary>

In [30]:
@torch.no_grad()
def diffusion_generate(
    model,
    tokenizer,
    prompt_text: str,
    max_new_tokens: int = 128,
    diffusion_steps: int = 64,
    temperature: float = 1.0,
    top_k: int = 0,
    record_steps: bool = True,
):
    model.eval()
    device = next(model.parameters()).device

    prompt_ids = tokenizer.encode(prompt_text, add_special_tokens=True)
    prompt_ids = torch.tensor(prompt_ids, dtype=torch.long, device=device).unsqueeze(0)  # [1, Lp]

    Lp = prompt_ids.size(1)
    L = min(cfg["context_length"], Lp + max_new_tokens)
    gen_len = L - Lp

    x = torch.full((1, L), MASK_ID, dtype=torch.long, device=device)
    x[:, :Lp] = prompt_ids[:, :Lp]

    fixed = torch.zeros((1, L), dtype=torch.bool, device=device)
    fixed[:, :Lp] = True

    attention_mask = torch.ones((1, L), dtype=torch.bool, device=device)

    frames = []

    def sample_from_logits(logits):
        if temperature != 1.0:
            logits = logits / temperature

        if top_k and top_k > 0:
            topk_vals, topk_idx = torch.topk(logits, k=top_k, dim=-1)
            filtered = torch.full_like(logits, float("-inf"))
            filtered.scatter_(-1, topk_idx, topk_vals)
            logits = filtered

        probs = F.softmax(logits, dim=-1)
        flat = probs.view(-1, probs.size(-1))
        sampled = torch.multinomial(flat, num_samples=1).view(1, L)
        sampled_prob = probs.gather(-1, sampled.unsqueeze(-1)).squeeze(-1)  # [1,L]
        return sampled, sampled_prob

    for s in range(diffusion_steps, 0, -1):
        t = torch.tensor([s], device=device, dtype=torch.long)
        logits = model(x, timesteps=t, attention_mask=attention_mask)
        sampled, conf = sample_from_logits(logits)

        update_pos = ~fixed
        x[update_pos] = sampled[update_pos]

        next_ratio = float(s - 1) / float(diffusion_steps)
        target_masks = int(math.ceil(gen_len * next_ratio))

        gen_positions = torch.arange(L, device=device) >= Lp
        candidates = gen_positions & (~fixed[0])
        cand_idx = torch.where(candidates)[0]

        if target_masks > 0 and cand_idx.numel() > 0:
            cand_conf = conf[0, cand_idx]
            k = min(target_masks, cand_idx.numel())
            _, low_idx = torch.topk(cand_conf, k=k, largest=False)
            remask_positions = cand_idx[low_idx]
            x[0, remask_positions] = MASK_ID

        if record_steps:
            decoded = tokenizer.decode(x[0].tolist())
            decoded = decoded.replace("[MASK]", "█")
            frames.append((s, decoded))

    final = tokenizer.decode(x[0].tolist())
    model.train()
    return final, frames

def chat_prompt(user_msg: str, system_msg: str = None) -> str:
    parts = []
    if system_msg:
        parts.append(f"<|system|>\n{system_msg}\n")
    parts.append(f"<|user|>\n{user_msg}\n")
    parts.append("<|assistant|>\n")
    return "".join(parts)

TEST_USER_PROMPT = "Once upon a time"
prompt_text = chat_prompt(TEST_USER_PROMPT)

final_text, frames = diffusion_generate(
    model=model,
    tokenizer=hf_tokenizer,
    prompt_text=prompt_text,
    max_new_tokens=128,
    diffusion_steps=cfg["diffusion_steps"],
    temperature=1.0,
    top_k=50,
    record_steps=True,
)

print("Final decoded (raw):\n")
print(final_text[:1000])
print("\nRecorded frames:", len(frames))

c:\Users\mouad\VSCode_Projects\.venv\Lib\site-packages\torch\nn\modules\transformer.py:505: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\NestedTensorImpl.cpp:182.)
  output = torch._nested_tensor_from_mask(


Final decoded (raw):

[BOS]<|user|>
Once upon a time
<|assistant|>
[EOS] he She his He, that short she and her so! story a it.. to to her She. that she. to. to was,. of and 
 a short. and. was. She he the to a. Lily day to They was.. a it very the.<|end|>. to Lily there a.<|user|> to the.
. the a, to and<|assistant|> a
. was her,
, of his he and little. in 
 the a.
 his., so was for
 the
., to were story. he and., the The
 was, and.! and time the

Recorded frames: 64


</details>

## Output rendering animation

In [57]:
from PIL import Image, ImageDraw, ImageFont
import numpy as np
import os

def text_to_image(text, width=800, height=400):
    img = Image.new("RGB", (width, height), color="black")
    draw = ImageDraw.Draw(img)

    local_app_data_path = os.environ["LOCALAPPDATA"]
    fonts_path = os.path.join(local_app_data_path, "Microsoft\Windows\Fonts", "LiberationMono-Regular.ttf")
    font = ImageFont.truetype(fonts_path, 15)    

    draw.text((10, 10), text[:500], fill="white", font=font)  
    return img

In [58]:
images = [text_to_image(f) for f in frames]

In [59]:
len(frames)

128

In [60]:
frames[-1]

'[BOS]<|user|>\nOnce upon a time.\n<|assistant|>\n<|end|>\n[EOS]\nWrite a.\n<|assistant|>\nThere was a boy named Tim. Tim lived in the park with his mom. He liked to catch catch in the park. One day, he saw a big bird in the sky. It was a pretty bird. Tim wanted to catch with the bird, but his mom said no.\n\nTim was very sad. He kept catch with the bird. He felt sad. His mom said, "Tim, you can\'t catch with the bird. It is mine."\n\nTim\'s mom. She said, "No, Tim. You cannot play with bird. You can\'t have it."\n\nTim was sad. He wanted to catch with the bird. He He wanted the bird too. He wanted to keep the bird. He asked his mom, "Can I keep the bird?" His mom said, "No, you can keep my. You can keep. will bite you."\n\nTim did not listen. He did not like the bird. He wanted to hurt his mom.\n\nThen, big bird saw the bird. The bird flew away. It flew to the tree. Tim and his mom were not happy. They played with the bird'

In [ ]:
# images[0].save(
#     "../inference_diffusion_animation.gif",
#     save_all=True,
#     append_images=images[1:],
#     duration=50,
# )

In [61]:
## add autoregressive lm animation to it
final_text = frames[-1]
ar_tokens = hf_tokenizer.tokenize(final_text)
ar_frames = []

for i in range(len(ar_tokens) + 1):
    visible = ar_tokens[:i]

    ar_frames.append(
        hf_tokenizer.convert_tokens_to_string(visible)
    )

In [62]:
len(ar_frames), len(frames)

(257, 128)

In [ ]:
# # resampling to make sure they have the same nbr frames
# idx = np.linspace(
#     0,
#     len(ar_frames) - 1,
#     len(frames)
# ).astype(int)

# ar_frames = [ar_frames[i] for i in idx]

In [65]:
len(ar_frames)

257

In [66]:
import textwrap

local_app_data_path = os.environ["LOCALAPPDATA"]
font_path = os.path.join(
    local_app_data_path,
    "Microsoft\\Windows\\Fonts",
    "LiberationMono-Regular.ttf"
)

font = ImageFont.truetype(font_path, 12)
title_font = ImageFont.truetype(font_path, 22)

def two_column_frame(diff_text, ar_text, diff_step=None, ar_step=None,
                     width=1400, height=600):

    img = Image.new("RGB", (width, height), "black")
    draw = ImageDraw.Draw(img)

    mid = width // 2

    # ---------------- TITLES (CENTERED) ----------------
    title1 = "Diffusion Language Model"
    title2 = "Autoregressive Language Model"

    b1 = draw.textbbox((0, 0), title1, font=title_font)
    b2 = draw.textbbox((0, 0), title2, font=title_font)

    w1 = b1[2] - b1[0]
    w2 = b2[2] - b2[0]

    draw.text(((mid - w1) // 2, 10), title1, fill="deepskyblue", font=title_font)
    draw.text((mid + (mid - w2) // 2, 10), title2, fill="orange", font=title_font)

    # ---------------- STEP COUNTERS ----------------
    if diff_step is not None:
        draw.text((20, 45), f"Denoising step {diff_step}", fill="gray", font=font)

    if ar_step is not None:
        draw.text((mid + 20, 45), f"Decoding iteration {ar_step}", fill="gray", font=font)

    # separator
    draw.line([(mid, 0), (mid, height)], fill=(60, 60, 60), width=2)

    # ---------------- WRAP TEXT ----------------
    def wrap_preserve_newlines(text, width_chars=90):
        lines = text.split("\n")
        wrapped_lines = []

        for line in lines:
            wrapped = textwrap.wrap(line, width=width_chars)

            # handle empty lines explicitly
            if len(wrapped) == 0:
                wrapped_lines.append("")
            else:
                wrapped_lines.extend(wrapped)

        return "\n".join(wrapped_lines)
    diff_text = wrap_preserve_newlines(diff_text)
    ar_text = wrap_preserve_newlines(ar_text)

    # ---------------- DRAW TEXT ----------------
    draw.multiline_text(
        (20, 80),
        diff_text,
        fill="white",
        font=font,
        spacing=4
    )

    draw.multiline_text(
        (mid + 20, 80),
        ar_text,
        fill="white",
        font=font,
        spacing=4
    )

    return img

In [67]:
n_frames = max(len(frames), len(ar_frames))

comparison_frames = []

for i in range(n_frames):

    diff_text = frames[i] if i < len(frames) else frames[-1]
    ar_text = ar_frames[i] if i < len(ar_frames) else ar_frames[-1]

    comparison_frames.append(
        two_column_frame(
            diff_text,
            ar_text,
            diff_step=i if i < len(frames) else len(frames) - 1,
            ar_step=i if i < len(ar_frames) else len(ar_frames) - 1
        )
    )

In [ ]:
comparison_frames[0].save(
    "../diffusion_autoregressive_anim.gif",
    save_all=True,
    append_images=comparison_frames[1:],
    duration=50
    )